In [1]:
# Data handling
import pandas as pd
import numpy as np

# NLP
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Machine learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Visualisation
import matplotlib.pyplot as plt

# Download NLP resources
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
# Load dataset
df = pd.read_csv("WhoDataSet.csv")

# Display first rows
df.head()

,episode_id,title,season,doctorid,is_special,is_earth,is_space,is_past,is_present,is_future,...,Has Sarah Jane Smith,Has Sontaran,Has Sophie,Has The Master,Has The Silent,Has The War Doctor,Has Weeping Angel,Has Winston Churchill,Has Yasmin Khan,Has Zygon
0,1.01,Rose,1,9,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,1.02,The End of the World,1,9,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,1.03,The Unquiet Dead,1,9,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1.04,Aliens of London,1,9,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,1.05,World War Three,1,9,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 164 entries, 0 to 163
Data columns (total 63 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   episode_id                   164 non-null    float64
 1   title                        164 non-null    str    
 2   season                       164 non-null    int64  
 3   doctorid                     164 non-null    int64  
 4   is_special                   164 non-null    int64  
 5   is_earth                     164 non-null    int64  
 6   is_space                     164 non-null    int64  
 7   is_past                      164 non-null    int64  
 8   is_present                   164 non-null    int64  
 9   is_future                    164 non-null    int64  
 10  is_outside_time              164 non-null    int64  
 11  Producer                     164 non-null    str    
 12  Director                     164 non-null    str    
 13  Writer                       16

In [5]:
df.columns

Index(['episode_id', 'title', 'season', 'doctorid', 'is_special', 'is_earth',
       'is_space', 'is_past', 'is_present', 'is_future', 'is_outside_time',
       'Producer', 'Director', 'Writer', 'Music', 'date', 'Aired_Fri',
       'Aired_Mon', 'Aired_Sat', 'Aired_Sun', 'Aired_Tue', 'Aired_Wed',
       'rating', 'views', 'Has 10', 'Has 11', 'Has 12', 'Has 13', 'Has 9',
       'Has Amy Pond', 'Has Bill', 'Has Clara', 'Has Cyberman', 'Has Dalek',
       'Has Danny Pink', 'Has Donna Noble', 'Has Grace', 'Has Graham O'Brien',
       'Has Jackie Tyler', 'Has Jenny Flint', 'Has Judoon',
       'Has Kate Lethbridge-Stewart', 'Has Madame Kovarian',
       'Has Madame Vastra', 'Has Martha Jones', 'Has Mickey Smith',
       'Has Nardole', 'Has Ood', 'Has Osgood', 'Has River Song', 'Has Rory',
       'Has Rose Tyler', 'Has Ryan Sinclair', 'Has Sarah Jane Smith',
       'Has Sontaran', 'Has Sophie', 'Has The Master', 'Has The Silent',
       'Has The War Doctor', 'Has Weeping Angel', 'Has Winston 

In [7]:
text_column = "text"

if text_column not in df.columns:
    print("Available columns:", df.columns.tolist())
else:
    documents = df[text_column].astype(str)

Available columns: ['episode_id', 'title', 'season', 'doctorid', 'is_special', 'is_earth', 'is_space', 'is_past', 'is_present', 'is_future', 'is_outside_time', 'Producer', 'Director', 'Writer', 'Music', 'date', 'Aired_Fri', 'Aired_Mon', 'Aired_Sat', 'Aired_Sun', 'Aired_Tue', 'Aired_Wed', 'rating', 'views', 'Has 10', 'Has 11', 'Has 12', 'Has 13', 'Has 9', 'Has Amy Pond', 'Has Bill', 'Has Clara', 'Has Cyberman', 'Has Dalek', 'Has Danny Pink', 'Has Donna Noble', 'Has Grace', "Has Graham O'Brien", 'Has Jackie Tyler', 'Has Jenny Flint', 'Has Judoon', 'Has Kate Lethbridge-Stewart', 'Has Madame Kovarian', 'Has Madame Vastra', 'Has Martha Jones', 'Has Mickey Smith', 'Has Nardole', 'Has Ood', 'Has Osgood', 'Has River Song', 'Has Rory', 'Has Rose Tyler', 'Has Ryan Sinclair', 'Has Sarah Jane Smith', 'Has Sontaran', 'Has Sophie', 'Has The Master', 'Has The Silent', 'Has The War Doctor', 'Has Weeping Angel', 'Has Winston Churchill', 'Has Yasmin Khan', 'Has Zygon']


In [10]:
df.head()

,episode_id,title,season,doctorid,is_special,is_earth,is_space,is_past,is_present,is_future,...,Has Sarah Jane Smith,Has Sontaran,Has Sophie,Has The Master,Has The Silent,Has The War Doctor,Has Weeping Angel,Has Winston Churchill,Has Yasmin Khan,Has Zygon
0,1.01,Rose,1,9,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,1.02,The End of the World,1,9,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,1.03,The Unquiet Dead,1,9,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1.04,Aliens of London,1,9,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,1.05,World War Three,1,9,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
print("documents:", "documents" in globals())
print("clean_documents:", "clean_documents" in globals())
print("X:", "X" in globals())

documents: False
clean_documents: False
X: False


In [13]:
df.head()

,episode_id,title,season,doctorid,is_special,is_earth,is_space,is_past,is_present,is_future,...,Has Sarah Jane Smith,Has Sontaran,Has Sophie,Has The Master,Has The Silent,Has The War Doctor,Has Weeping Angel,Has Winston Churchill,Has Yasmin Khan,Has Zygon
0,1.01,Rose,1,9,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,1.02,The End of the World,1,9,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,1.03,The Unquiet Dead,1,9,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1.04,Aliens of London,1,9,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,1.05,World War Three,1,9,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [16]:
print(tfidf)

TfidfVectorizer(max_features=2000, ngram_range=(1, 2))


In [17]:
df.to_csv(
    "WhoDataSet_clustered.csv",
    index=False
)

print("Saved successfully")

Saved successfully
